# To do

Comorbidities
1. add KSADS comorbidities to the baseline_cohort
2. add child-based KSADS diagnoses for comorbidities
3. summarize comorbidities (eg. all suicidal ideation, all mdd)
4. consider creating 'strict' diagnoses that also merge in parent cbcl or teacher bpm

ADHD-subtypes
1. find all relevant variables
2. create subtype definitions

cognitive + education
1. find education variables and encode them
2. add age-adjusted nih toolbox scores

# Resources

Ideas for ADHD definition: 
    Paper: https://pmc.ncbi.nlm.nih.gov/articles/PMC9677584
    Supplement: https://pmc.ncbi.nlm.nih.gov/articles/instance/9677584/bin/NIHMS1851354-supplement-Online_Supplement.pdf

Ideas for ADHD medication:
    Paper: https://pmc.ncbi.nlm.nih.gov/articles/PMC11541585/#MOESM1
    RxNorm Drug Names: https://mor.nlm.nih.gov/RxNav/ 

ABCD resources:
    Documentation: https://docs.abcdstudy.org/latest/documentation/non_imaging/ab.html
    Variable browser: https://abcd.deapscience.com/#/home


# Import Libraries

In [1]:
# Standard library imports
import importlib
import re
import random

# Packages installed in conda env imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Local imports
from utils.get_abcd_data import (
    get_demo_file, 
    get_ksads_parent_file, 
    get_bpm_teacher_file,
    get_cbcl_parent_file,
    get_family_history_parent_file, 
    get_medications_inventory_parent_file, 
    get_developmental_history_parent_file
    )

# Set up

In [2]:
#set seed

seed = 123
np.random.seed(seed)
random.seed(seed)

# Demographics

In [3]:
#get demographics file 
demo_df = pd.read_csv(get_demo_file(),low_memory=False)

# make a copy for editing
demo_edited = demo_df.copy()

# Subset to baseline only
demo_edited = demo_edited[demo_edited['eventname'] == 'baseline_year_1_arm_1']

In [4]:
# edit age

demo_edited = demo_edited.rename(columns={"demo_brthdat_v2": "age"})

In [5]:
# edit sex

#drop the two intersex male rows since the sample is too small to analyze
demo_edited = demo_edited[demo_edited.demo_sex_v2 != 3.0]

#Set 1.0 as "Male" and 0.0 as "Female" in the sex column
demo_edited['sex'] = demo_edited['demo_sex_v2'].map({1.0: 'Male', 2.0: 'Female'})

#drop demo_sex_v2 column
demo_edited = demo_edited.drop(columns=["demo_sex_v2"])

In [6]:
#edit race_ethnicity

# map race/ethnicity values to their corresponding labels
demo_edited['race_ethnicity'] = demo_edited['race_ethnicity'].map({1.0: 'White', 2.0: 'Black', 3.0: 'Hispanic', 4.0: 'Asian', 5.0: 'Other'})

In [7]:
# edit income to encoding below/above federal poverty line
#for 2017, the poverty line for a household of 1 is 12060 and 4180 is added for each additional member


# Map codes to approximate midpoints of ranges captured by demo_comb_income_v2
income_map = {
    1: 2500,
    2: 8500,
    3: 14000,
    4: 20500,
    5: 30000,
    6: 42500,
    7: 62500,
    8: 87500,
    9: 150000,
    10: 200000, #minimum (since range >= 200,000)
    777: np.nan,
    999: np.nan
}

demo_edited['income_estimate'] = demo_edited['demo_comb_income_v2'].map(income_map)

# Set demo_roster_v2 to NaN when:
# 1. demo_roster_v2_refuse is 777 or 999
# 2. household size < 2 (implausible)
# 3. household size > 15 (implausible)
demo_edited.loc[
    (demo_edited["demo_roster_v2_refuse"].isin([777, 999])) |
    (demo_edited["demo_roster_v2"] < 2) |
    (demo_edited["demo_roster_v2"] > 15),
    "demo_roster_v2"
] = np.nan

# Poverty line function
def calc_poverty_line(size):
    return 12060 + 4180 * (size - 1)

# Compute poverty line
demo_edited["fpl_2017_by_household_size"] = demo_edited["demo_roster_v2"].apply(calc_poverty_line)

# Determine FPL status, but only where both income and household size are valid

# Initialize with NaN (object type)
demo_edited["fpl_2017_status"] = np.nan

# Only assign when both income and household size are available
mask_valid_income_fpl = demo_edited["income_estimate"].notna() & demo_edited["fpl_2017_by_household_size"].notna()

demo_edited.loc[mask_valid_income_fpl, "fpl_2017_status"] = np.where(
    demo_edited.loc[mask_valid_income_fpl, "income_estimate"] > demo_edited.loc[mask_valid_income_fpl, "fpl_2017_by_household_size"],
    "above FPL",
    "below FPL"
)

#drop demo_comb_income_v2, demo_roster_v2, demo_roster_v2_refuse, income_estimate, fpl_2017_by_household_size columns
demo_edited = demo_edited.drop(columns=['demo_comb_income_v2', 'demo_roster_v2', 'demo_roster_v2_refuse', 'income_estimate', 'fpl_2017_by_household_size'])

/var/folders/gr/h8xjwb8j6xv0ywn4mxk1sm_r0000gn/T/ipykernel_53314/3567541647.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['above FPL' 'above FPL' 'above FPL' ... 'above FPL' 'below FPL'
 'below FPL']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  demo_edited.loc[mask_valid_income_fpl, "fpl_2017_status"] = np.where(


In [8]:
#create highest parental education level variable

# Clean missing / refusal codes
demo_edited.loc[demo_edited["demo_prnt_ed_v2"].isin([777, 999]), "demo_prnt_ed_v2"] = np.nan
demo_edited.loc[demo_edited["demo_prtnr_ed_v2"].isin([777, 999]), "demo_prtnr_ed_v2"] = np.nan
demo_edited.loc[demo_edited["demo_prnt_prtnr_v2"] == 777, "demo_prnt_prtnr_v2"] = np.nan

# Shortcut variables
p_ed = demo_edited["demo_prnt_ed_v2"]
pt_ed = demo_edited["demo_prtnr_ed_v2"]
hp = demo_edited["demo_prnt_prtnr_v2"]

# Define “has degree” flags
has_college_parent = p_ed >= 16
has_college_partner = pt_ed >= 16

# 1️⃣ College degree if either parent or partner has one
college = has_college_parent | has_college_partner

# 2️⃣ Explicitly below college if:
#     - parent known and <16
#     - and either: (a) no partner (==2), OR (b) partner known and also <16
below = (
    p_ed.notna() & (p_ed < 16) &
    ((hp == 2) | (pt_ed.notna() & (pt_ed < 16)))
)

# 3️⃣ Combine into one variable
demo_edited["highest_parental_ed"] = np.select(
    [college, below],
    [1, 0],
    default=np.nan
)

# Optional label
demo_edited["highest_parental_ed"] = demo_edited["highest_parental_ed"].map({
    1: "College degree or above",
    0: "Below college degree"
})



In [9]:
# make cohort with only desired columns

#desired columns
demo_cols_to_keep = ["src_subject_id", "age", "sex", "race_ethnicity", "fpl_2017_status", "highest_parental_ed"]

baseline_cohort = demo_edited[demo_cols_to_keep].copy()

# Mental health diagnoses

In [10]:
#get parent ksads
ksads_p_df = pd.read_csv(get_ksads_parent_file(),low_memory=False)

#get teacher bpm
bpm_t_df = pd.read_csv(get_bpm_teacher_file(),low_memory=False)

#get parent cbcl
cbcl_p_df = pd.read_csv(get_cbcl_parent_file(),low_memory=False)

# Merge all three DataFrames on the same keys
mental_health_edited = (
    ksads_p_df
    .merge(bpm_t_df, on=["src_subject_id", "eventname"], how="outer")
    .merge(cbcl_p_df, on=["src_subject_id", "eventname"], how="outer")
)

# Subset to baseline only
mental_health_edited = mental_health_edited[mental_health_edited['eventname'] == 'baseline_year_1_arm_1']

In [11]:
#ADHD diagnosis

#ksads
# ksads_14_853_p	Diagnosis: Attention-deficit/hyperactivity disorder - Present [Parent]
# ksads_14_855_p 	Diagnosis: Attention-deficit/hyperactivity disorder - Partial remission [Parent]
# ksads_14_856_p	Diagnosis: Unspecified Attention-deficit/hyperactivity disorder (F90.9) [Parent]

#bpm teacher form
# bpm_t_scr_attention_t   Brief Problem Monitor [Teacher] (Attention): T-score [Validation: No more than 0 missing or declined]

#cbcl
# cbcl_scr_syn_attention_t      Child Behavior Checklist [Parent] (Syndrome Scale - Attention problems): T-score [Validation: No more than 0 missing or declined]
# cbcl_scr_dsm5_adhd_t          Child Behavior Checklist [Parent] (DSM-5 Oriented Scale - ADHD): T-score [Validation: No more than 0 missing or declined]


#ksads-only based diagnosis
mental_health_edited["adhd_ksads"] = np.where(
    (
        (mental_health_edited["ksads_14_853_p"] == 1) |
        (mental_health_edited["ksads_14_855_p"] == 1) |
        (mental_health_edited["ksads_14_856_p"] == 1)
    ),
    "yes",
    "no"
)


#ksads + clinical range on teacher bmp or parent cbcl attention/adhd subscales
mental_health_edited["adhd_strict"] = np.where(
    (
        (mental_health_edited["ksads_14_853_p"] == 1) |
        (mental_health_edited["ksads_14_855_p"] == 1) |
        (mental_health_edited["ksads_14_856_p"] == 1)
    ) &
    (
        (mental_health_edited["cbcl_scr_syn_attention_t"] >= 65) |
        (mental_health_edited["cbcl_scr_dsm5_adhd_t"] >= 65) |
        (mental_health_edited["bpm_t_scr_attention_t"] >= 65)
    ),
    "yes",
    "no"
)


In [12]:
#other KSADS-based diagnoses

# Dictionary of KSADS variable names and their clean summary labels
# only using present and full diagnoses here
dx_map = {
    # Present diagnoses
    "ksads_4_828_p": "delusions",
    "ksads_4_826_p": "hallucinations",
    "ksads_6_859_p": "agoraphobia",
    "ksads_2_831_p": "bipolar_I_current_depressed",
    "ksads_2_832_p": "bipolar_I_current_hypomanic",
    "ksads_2_830_p": "bipolar_I_current_manic",
    "ksads_2_834_p": "bipolar_I_recent_depressed",
    "ksads_2_833_p": "bipolar_I_recent_manic",
    "ksads_2_836_p": "bipolar_II_current_depressed",
    "ksads_2_835_p": "bipolar_II_current_hypomanic",
    "ksads_2_837_p": "bipolar_II_recent_hypomanic",
    "ksads_2_838_p": "bipolar_unspecified",
    "ksads_16_898_p": "conduct_disorder_adolescent_onset",
    "ksads_16_897_p": "conduct_disorder_childhood_onset",
    "ksads_1_840_p": "major_depressive_disorder",
    "ksads_1_843_p": "persistent_depressive_disorder",
    "ksads_1_846_p": "depressive_disorder_unspecified",
    "ksads_3_848_p": "disruptive_mood_dysregulation_disorder",
    "ksads_13_929_p": "anorexia_binge_purge",
    "ksads_13_932_p": "anorexia_restricting",
    "ksads_13_941_p": "atypical_anorexia",
    "ksads_13_938_p": "binge_eating",
    "ksads_13_944_p": "binge_eating_lowfreq",
    "ksads_13_935_p": "bulimia",
    "ksads_13_942_p": "bulimia_lowfreq",
    "ksads_10_869_p": "generalized_anxiety_disorder",
    "ksads_24_967_p": "homicidal",
    "ksads_11_917_p": "obsessive_compulsive_disorder",
    "ksads_15_901_p": "oppositional_defiant_disorder",
    "ksads_5_857_p": "panic_disorder",
    "ksads_9_867_p": "specific_phobia",
    "ksads_4_849_p": "associated_psychotic_symptoms",
    "ksads_4_851_p": "unspecified_schizophrenia_spectrum",
    "ksads_21_921_p": "post_traumatic_stress_disorder",
    "ksads_7_861_p": "separation_anxiety",
    "ksads_22_969_p": "sleep_problems",
    "ksads_8_863_p": "social_anxiety_disorder",
    "ksads_23_949_p": "suicidal_intent_active",
    "ksads_23_948_p": "suicidal_method_active",
    "ksads_23_950_p": "suicidal_plan_active",
    "ksads_23_947_p": "suicidal_nonspecific_active",
    "ksads_23_953_p": "aborted_suicide_attempt",
    "ksads_23_952_p": "interrupted_suicide_attempt",
    "ksads_23_954_p": "suicide_attempt",
    "ksads_23_951_p": "suicidal_preparatory_behavior",
    "ksads_23_946_p": "suicidal_ideation_passive",
    "ksads_23_945_p": "self_injury_nonsuicidal"
}

# Recode and create new columns
for var, new_col in dx_map.items():
    mental_health_edited[new_col] = mental_health_edited[var].replace({
        555: np.nan,
        888: "not present",
        0: "not present",
        1: "present"
    })

In [13]:
# no current or past diagnosis

# --- list of all diagnosis variables you provided ---
dx_vars = [
    # ADHD
    "ksads_14_854_p", "ksads_14_853_p", "ksads_14_855_p", "ksads_14_856_p",

    # Partial remission
    "ksads_1_841_p", "ksads_1_844_p", "ksads_13_930_p", "ksads_13_933_p",
    "ksads_13_939_p", "ksads_13_937_p",

    # Subthreshold
    "ksads_6_908_p", "ksads_18_903_p", "ksads_10_914_p", "ksads_10_913_p",
    "ksads_11_920_p", "ksads_11_919_p", "ksads_5_907_p", "ksads_5_906_p",
    "ksads_21_924_p", "ksads_21_923_p", "ksads_7_910_p", "ksads_7_909_p",
    "ksads_8_912_p", "ksads_8_911_p",

    # Past diagnoses
    "ksads_4_829_p", "ksads_4_827_p", "ksads_6_860_p", "ksads_2_839_p",
    "ksads_16_900_p", "ksads_16_899_p", "ksads_1_842_p", "ksads_1_845_p",
    "ksads_1_847_p", "ksads_13_931_p", "ksads_13_934_p", "ksads_13_971_p",
    "ksads_13_940_p", "ksads_13_972_p", "ksads_13_936_p", "ksads_13_943_p",
    "ksads_10_870_p", "ksads_24_968_p", "ksads_11_918_p", "ksads_15_902_p",
    "ksads_5_858_p", "ksads_9_868_p", "ksads_4_850_p", "ksads_4_852_p",
    "ksads_21_922_p", "ksads_7_862_p", "ksads_22_970_p", "ksads_8_864_p",
    "ksads_23_960_p", "ksads_23_959_p", "ksads_23_961_p", "ksads_23_958_p",
    "ksads_23_964_p", "ksads_23_963_p", "ksads_23_965_p", "ksads_23_962_p",
    "ksads_23_957_p", "ksads_23_956_p", 

    # Present diagnoses
    "ksads_4_828_p", "ksads_4_826_p", "ksads_6_859_p", "ksads_2_831_p",
    "ksads_2_832_p", "ksads_2_830_p", "ksads_2_834_p", "ksads_2_833_p",
    "ksads_2_836_p", "ksads_2_835_p", "ksads_2_837_p", "ksads_2_838_p",
    "ksads_16_898_p", "ksads_16_897_p", "ksads_1_840_p", "ksads_1_843_p",
    "ksads_1_846_p", "ksads_3_848_p", "ksads_13_929_p", "ksads_13_932_p",
    "ksads_13_941_p", "ksads_13_938_p", "ksads_13_944_p", "ksads_13_935_p",
    "ksads_13_942_p", "ksads_10_869_p", "ksads_24_967_p", "ksads_11_917_p",
    "ksads_15_901_p", "ksads_5_857_p", "ksads_9_867_p", "ksads_4_849_p",
    "ksads_4_851_p", "ksads_21_921_p", "ksads_7_861_p", "ksads_22_969_p",
    "ksads_8_863_p", "ksads_23_949_p", "ksads_23_948_p", "ksads_23_950_p",
    "ksads_23_947_p", "ksads_23_953_p", "ksads_23_952_p", "ksads_23_954_p",
    "ksads_23_951_p", "ksads_23_946_p", "ksads_23_945_p"
]

# Create the no_diagnosis column with the new rule
mental_health_edited["no_diagnosis"] = np.where(
    mental_health_edited[dx_vars].apply(lambda x: x.isin([0, 888]).all(), axis=1),
    "yes",  # all are 0, 888 (not asked due to branching)
    "no"    # any 1 or 555 → has/possible diagnosis
)

In [14]:
# --- Keep only columns for merging ---
mh_merge = mental_health_edited[["src_subject_id", "adhd_ksads", "adhd_strict", "no_diagnosis"]].copy()

# --- Merge into baseline_cohort by src_subject_id ---
baseline_cohort = baseline_cohort.merge(mh_merge, on='src_subject_id', how="left")

# ADHD-subtypes

# Parental mental health diagnosis

In [15]:
#get family history
fhx_p_df = pd.read_csv(get_family_history_parent_file(),low_memory=False)

# make a copy for editing
fhx_edited = fhx_p_df.copy()

# Subset to baseline only
fhx_edited = fhx_edited[fhx_edited['eventname'] == 'baseline_year_1_arm_1']

In [16]:
#father variables: fam_history_q13a_suicide, fam_history_q11a_professional, fam_history_q6a_depression, fam_history_q9a_trouble,
#fam_history_q8a_visions, fam_history_q12a_hospitalized, fam_history_q7a_mania, fam_history_q10a_nerves
#not including drugs and alcohol here for now, but this can be added

# --- Father-specific variables in fhx_edited ---
father_vars = [
    "fam_history_q13a_suicide",
    "fam_history_q11a_professional",
    "fam_history_q6a_depression",
    "fam_history_q9a_trouble",
    "fam_history_q8a_visions",
    "fam_history_q12a_hospitalized",
    "fam_history_q7a_mania",
    "fam_history_q10a_nerves"
]

# Corresponding yes/no variables
yesno_vars = [
    "fam_history_13_yes_no",
    "fam_history_11_yes_no",
    "fam_history_6_yes_no",
    "fam_history_9_yes_no",
    "fam_history_8_yes_no",
    "fam_history_12_yes_no",
    "fam_history_7_yes_no",
    "fam_history_10_yes_no"
]

# --- Step 1: Force father variables = 0 where *_yes_no == 0 ---
for f_var, yesno_var in zip(father_vars, yesno_vars):
    fhx_edited.loc[fhx_edited[yesno_var] == 0, f_var] = 0

# --- Step 2: Create binary father_mh_issues variable ---
fhx_edited["father_mh_issues"] = np.nan

# Any father var == 1 → 1
fhx_edited.loc[
    (fhx_edited[father_vars] == 1).any(axis=1),
    "father_mh_issues"
] = 1

# All father vars == 0 → 0
fhx_edited.loc[
    (fhx_edited[father_vars] == 0).all(axis=1),
    "father_mh_issues"
] = 0

# --- Step 3: Convert to readable string labels ---
fhx_edited["father_mh_issues"] = fhx_edited["father_mh_issues"].map({
    0: "no issue",
    1: "mh issue present"
})


In [17]:
#mother variables: fam_history_q13d_suicide, fam_history_q11d_professional, fam_history_q6d_depression, fam_history_q9d_trouble,
#fam_history_q8d_visions, fam_history_q12d_hospitalized, fam_history_q7d_mania, fam_history_q10d_nerves
#not including drugs and alcohol here for now, but this can be added


# --- Mother-specific variables in fhx_edited ---
mother_vars = [
    "fam_history_q13d_suicide",
    "fam_history_q11d_professional",
    "fam_history_q6d_depression",
    "fam_history_q9d_trouble",
    "fam_history_q8d_visions",
    "fam_history_q12d_hospitalized",
    "fam_history_q7d_mania",
    "fam_history_q10d_nerves"
]

# Corresponding yes/no variables (same pattern as before)
yesno_vars = [
    "fam_history_13_yes_no",
    "fam_history_11_yes_no",
    "fam_history_6_yes_no",
    "fam_history_9_yes_no",
    "fam_history_8_yes_no",
    "fam_history_12_yes_no",
    "fam_history_7_yes_no",
    "fam_history_10_yes_no"
]

# --- Step 1: Force mother vars = 0 where *_yes_no == 0 ---
for m_var, yesno_var in zip(mother_vars, yesno_vars):
    fhx_edited.loc[fhx_edited[yesno_var] == 0, m_var] = 0

# --- Step 2: Create binary mother_mh_issues variable ---
fhx_edited["mother_mh_issues"] = np.nan

# Any mother var == 1 → 1
fhx_edited.loc[
    (fhx_edited[mother_vars] == 1).any(axis=1),
    "mother_mh_issues"
] = 1

# All mother vars == 0 → 0
fhx_edited.loc[
    (fhx_edited[mother_vars] == 0).all(axis=1),
    "mother_mh_issues"
] = 0

# --- Step 3: Convert to readable string labels ---
fhx_edited["mother_mh_issues"] = fhx_edited["mother_mh_issues"].map({
    0: "no issue",
    1: "mh issue present"
})



In [18]:
# --- Keep only columns for merging ---
father_mother_mh_merge = fhx_edited[["src_subject_id", "father_mh_issues", "mother_mh_issues"]].copy()

# --- Merge into baseline_cohort by src_subject_id ---
baseline_cohort = baseline_cohort.merge(father_mother_mh_merge, on='src_subject_id', how="left")

# Medication

In [19]:
#get medications inventory
meds_p_df = pd.read_csv(get_medications_inventory_parent_file(),low_memory=False)

# make a copy for editing
meds_edited = meds_p_df.copy()

# Subset to baseline only
meds_edited = meds_edited[meds_edited['eventname'] == 'baseline_year_1_arm_1']


In [20]:
#ADHD medications

# stimulant medication: amphetamine/Adderall/Adzenys/Dyanavel/Evekeo/Mydayis; 
# Methylphenidate/Adhansia/Aptensio/Concerta/Cotempla/Daytrana/Jornay/Metadate/Methylin/QuilliChew/Quillivant/Relexxii/Ritalin;
# dextroamphetamine/Adderall/Dexedrine/Mydayis/ProCentra/Xelstrym/Zenzedi;
# dexmethylphenidate/Azstarys/Focalin;
# lisdexamfetamine/Vyvanse

# non-stimulant medication: aomoxetine/Strattera, guanfacine/Intuniv, viloxazine/Qelbree, clonidine/Catapres/Clorpres/Duraclon/Kapvay/Nexiclon/Onyda


# --- Step 1: Define your ADHD medication keywords ---
adhd_keywords = [
    # stimulant medications
    "amphetamine", "adderall", "adzenys", "dyanavel", "evekeo", "mydayis",
    "methylphenidate", "adhansia", "aptensio", "concerta", "cotempla", "daytrana",
    "jornay", "metadate", "methylin", "quillichew", "quillivant", "relexxii",
    "ritalin", "dextroamphetamine", "dexedrine", "procentra", "xelstrym", "zenzedi",
    "dexmethylphenidate", "azstarys", "focalin", "lisdexamfetamine", "vyvanse",
    # non-stimulant medications
    "atomoxetine", "strattera", "guanfacine", "intuniv", "viloxazine", "qelbree",
    "clonidine", "catapres", "clorpres", "duraclon", "kapvay", "nexiclon", "onyda"
]

# Combine into one regex pattern (case-insensitive)
adhd_pattern = re.compile("|".join(adhd_keywords), flags=re.IGNORECASE)

# --- Step 2: Identify medication name columns ---
rx_cols = [c for c in meds_edited.columns if c.endswith("_rxnorm_p")]

# --- Step 3: Detect ADHD medications anywhere in those columns ---
# create a boolean Series: True if any ADHD med is present in a row
has_adhd_med = meds_edited[rx_cols].astype(str).apply(
    lambda col: col.str.contains(adhd_pattern, na=False)
).any(axis=1)

# --- Step 4: Apply brought_medications logic ---
def classify_adhd_med(row):
    brought = row["brought_medications"]

    if pd.isna(brought) or brought in [1, 2, 4]:
        return np.nan
    elif has_adhd_med.loc[row.name]:
        return "yes"
    elif brought in [0, 3]:
        return "no"
    else:
        return np.nan

meds_edited["adhd_med"] = meds_edited.apply(classify_adhd_med, axis=1)


In [21]:
# mood/anxiety medications

#SSRIs: Zoloft/sertraline, Lexapro/escitalopram, Celexa/citalopram, Fluoxetine/PROzac/RECONCILE/Symbyax, Fluvoxamine, Paroxetine/Brisdelle/Paxil/Pexeva
#SNRIs: Desvenlafaxine/Khedezla/Pristiq, Duloxetine/Cymbalta/Drizalma/Irenka, Venlafaxine/Effexor, Levomilnacipran/Fetzima, Milnacipran/Savella
#atypical antidepressants: Mirtazapine/Mirataz/Remeron, bupropion/Wellbutrin
#missing: Vilazodone/Viibryd, Vortioxetine/Trintellix, buspirone/Bucapsol/Buspar
#Serotonin antagonist and reuptake inhibitors: Trazodone/Raldesy, Nefazodone
#MAOis: Tranylcypromine/Parnate, Phenelzine/Nardil, Selegiline/Anipryl/Eldepryl/Emsam/Zelapar, Isocarboxazid/Marplan
#TCAs: Nortriptyline/Pamelor, Desipramine/Norpramin, Protriptyline, Amoxapine, Amitriptyline/Elavil, Clomipramine/Anafranil/Calmera/Caniquell/Clomicalm, Doxepin/Prudoxin/Silenor/Zonalon, Imipramine, Trimipramine
#mood stabilizers: lithium, Valproic acid/valproate/Depakote/divalproex, lamotrigine/LaMICtal/Subvenite, quetiapine/Seroquel, aripiprazole/Abilify/Opipza, olanzapine/Lybalvi/Symbyax/ZyPREXA, carbamazepine/Carbatrol/Epitol/Equetro/Tegretol, oxcarbazepine/Oxtellar/Trileptal


# --- Step 1: Define mood & anxiety medication keywords ---
mood_anx_keywords = [
    # SSRIs
    "zoloft", "sertraline", "lexapro", "escitalopram", "celexa", "citalopram",
    "fluoxetine", "prozac", "reconcile", "symbyax", "fluvoxamine",
    "paroxetine", "brisdelle", "paxil", "pexeva",
    # SNRIs
    "desvenlafaxine", "khedezla", "pristiq", "duloxetine", "cymbalta",
    "drizalma", "irenka", "venlafaxine", "effexor", "levomilnacipran",
    "fetzima", "milnacipran", "savella",
    # Atypical antidepressants
    "mirtazapine", "mirataz", "remeron", "bupropion", "wellbutrin",
    # Others (non–class-specific)
    "vilazodone", "viibryd", "vortioxetine", "trintellix", "buspirone",
    "bucapsol", "buspar",
    # Serotonin antagonist and reuptake inhibitors
    "trazodone", "raldesy", "nefazodone",
    # MAOIs
    "tranylcypromine", "parnate", "phenelzine", "nardil", "selegiline",
    "anipryl", "eldepryl", "emsam", "zelapar", "isocarboxazid", "marplan",
    # TCAs
    "nortriptyline", "pamelor", "desipramine", "norpramin", "protriptyline",
    "amoxapine", "amitriptyline", "elavil", "clomipramine", "anafranil",
    "calmera", "caniquell", "clomicalm", "doxepin", "prudoxin", "silenor",
    "zonalon", "imipramine", "trimipramine",
    # Mood stabilizers / atypical antipsychotics
    "lithium", "valproic acid", "valproate", "depakote", "divalproex",
    "lamotrigine", "lamictal", "subvenite", "quetiapine", "seroquel",
    "aripiprazole", "abilify", "opipza", "olanzapine", "lybalvi", "symbyax",
    "zyprexa", "carbamazepine", "carbatrol", "epitol", "equetro", "tegretol",
    "oxcarbazepine", "oxtellar", "trileptal"
]

# Compile regex pattern
mood_anx_pattern = re.compile("|".join(mood_anx_keywords), flags=re.IGNORECASE)

# --- Step 2: Identify all medication columns ending in _rxnorm_p ---
rx_cols = [c for c in meds_edited.columns if c.endswith("_rxnorm_p")]

# --- Step 3: Detect if any of these meds appear in any _rxnorm_p column ---
has_mood_anx_med = meds_edited[rx_cols].astype(str).apply(
    lambda col: col.str.contains(mood_anx_pattern, na=False)
).any(axis=1)

# --- Step 4: Apply brought_medications logic ---
def classify_mood_anx_med(row):
    brought = row["brought_medications"]

    if pd.isna(brought) or brought in [1, 2, 4]:
        return np.nan
    elif has_mood_anx_med.loc[row.name]:
        return "yes"
    elif brought in [0, 3]:
        return "no"
    else:
        return np.nan

meds_edited["mood_anx_med"] = meds_edited.apply(classify_mood_anx_med, axis=1)


In [22]:
# --- Keep only columns for merging ---
meds_merge = meds_edited[["src_subject_id", "adhd_med", "mood_anx_med"]].copy()

# --- Merge into baseline_cohort by src_subject_id ---
baseline_cohort = baseline_cohort.merge(meds_merge, on='src_subject_id', how="left")

# Infant temperamental profile

In [23]:
#HBN:
#colic, difficult to soothe, sleeping difficulties, problems with social relatedness
#overly sensitive to sound, eating difficulties, slow to warm up
#baby limp/stiff, baby did not enjoy body contact

#unclear how to match these, I don't think there is anything similar in ABCD

# Language and motor skills

In [24]:
#get developmental history
developmental_history_p_df = pd.read_csv(get_developmental_history_parent_file(),low_memory=False)

# make a copy for editing
devhx_edited = developmental_history_p_df.copy()

# Subset to baseline only
devhx_edited = devhx_edited[devhx_edited['eventname'] == 'baseline_year_1_arm_1']

In [25]:
#HBN: months at which spoke first word, named objects, spoke 2/3 words

# devhx_19d_p	At approximately what age was this child first able to do each of the following?: Say their first word
# devhx_21_p	Would you say their speech development was earlier, average, or later than most other children?

# Create new column first_word
devhx_edited["first_word"] = devhx_edited["devhx_19d_p"]

# Define mapping
speech_map = {
    1: "Much earlier",
    2: "Somewhat earlier",
    3: "About average",
    4: "Somewhat later",
    5: "Much later"
}

# Create new column with mapped labels
devhx_edited["speech_dev"] = devhx_edited["devhx_21_p"].map(speech_map)

# Replace 999 ("Don't know") with NaN
devhx_edited.loc[devhx_edited["devhx_21_p"] == 999, "speech_dev"] = np.nan


In [26]:
#HBN: months at which held head up, sat without help, crawled, stood, cruised, walked, fed self, ran, rode a tricycle, dressed self, tied shoes

# devhx_19a_p	At approximately what age was this child first able to do each of the following?: Roll over

# devhx_19b_p	At approximately what age was this child first able to do each of the following?: Sit without assistance

# devhx_19c_p	At approximately what age was this child first able to do each of the following?: Walk without assistance

# devhx_20_p	Would you say their motor development (sitting, crawling, walking) was earlier, average, or later than most other children?

# Create new column roll_over
devhx_edited["roll_over"] = devhx_edited["devhx_19a_p"]

# Create new column sit
devhx_edited["sit"] = devhx_edited["devhx_19b_p"]

# Create new column walk
devhx_edited["walk"] = devhx_edited["devhx_19c_p"]

# Define mapping
motor_map = {
    1: "Much earlier",
    2: "Somewhat earlier",
    3: "About average",
    4: "Somewhat later",
    5: "Much later"
}

# Create new column with mapped labels
devhx_edited["motor_dev"] = devhx_edited["devhx_20_p"].map(motor_map)

# Replace 999 ("Don't know") with NaN
devhx_edited.loc[devhx_edited["devhx_20_p"] == 999, "motor_dev"] = np.nan


In [27]:
# --- Keep only columns for merging ---
dev_merge = devhx_edited[["src_subject_id", "first_word", "speech_dev", "roll_over", "sit", "walk", "motor_dev"]].copy()

# --- Merge into baseline_cohort by src_subject_id ---
baseline_cohort = baseline_cohort.merge(dev_merge, on='src_subject_id', how="left")

# Educational Attainment

In [28]:
#WIAT, listening comprehension (oral discourse), listening comprehension (receptive vocabulary),
#math problem solving, numerical operations, reading comprehension, spelling, word reading, pseudo-word decoding

# Cognitive Assessment

In [ ]:
#age-corrected NIH toolbox tasks: Flanker, list and card sorting, etc.